# Tome S1, Chapter 2: Determinants as Volume (Python companion)

Exercises first, solutions at the end. We measure polygons with the shoelace formula, build compound matrices and test the Cauchy–Binet theorem, run Dodgson's condensation in exact arithmetic, verify the Lindström–Gessel–Viennot lemma by brute force, count lozenge tilings three ways, sample a large random tiling to see the arctic circle, and count alternating sign matrices. Only `numpy`, `matplotlib` and the standard library are needed.

*Before each solution, predict what the output will look like.*

In [ ]:
import itertools, math, random
from fractions import Fraction
from math import comb
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import PolyCollection
rng = np.random.default_rng(2); random.seed(2)

## Exercises

**P2.1 (★) The shoelace formula.** Implement the formula of Example 2.1 and check it on the polygon of Exercise 2.1 (area $20$). Then generate a random star-shaped polygon with $30$ vertices and compare the formula with a Monte Carlo estimate of its area.

**P2.2 (★★) Compound matrices.** Write `compound(A, k)`, the matrix $C_k(A)$ of Definition 2.3. Check the Cauchy–Binet theorem $C_k(AB)=C_k(A)C_k(B)$ on random rectangular matrices, check Corollary 2.6 ($\det M^{\mathsf T}M=\sum_I\det(M_I)^2$), and check that the eigenvalues of $C_k(A)$ are the products of $k$ eigenvalues of $A$ (Exercise 2.5).

**P2.3 (★) Dodgson's condensation.** Implement Corollary 2.10 with exact fractions. Test it on $200$ random integer matrices against an exact determinant. On random $0/1$ matrices of size $6$, how often does the algorithm hit a zero denominator? Repair it by condensing $A+tI$ with $t$ a large random integer, a cheap version of the polynomial trick (Exercise 2.7): $\det(A+tI)$ is then a value of a polynomial, and interpolating $n+1$ such values recovers $\det A$.

**P2.4 (★★) The LGV lemma by brute force.** In the grid $\{0,\dots,4\}^2$ with north and east steps, choose sources $A_1=(0,1)$, $A_2=(1,0)$ and sinks $B_1=(3,4)$, $B_2=(4,3)$, and also a less symmetric choice of three sources and three sinks. Enumerate all path systems, and check that $\det(e(A_i,B_j))$ equals the signed number of non-intersecting systems (Theorem 2.11).

**P2.5 (★★) Counting lozenge tilings three ways.** Compute $T(a,b,c)$ (a) with the determinant of Theorem 2.14, (b) with MacMahon's product, and (c) by brute force, enumerating the plane partitions in an $a\times b$ array with entries $\le c$ that decrease along rows and columns. Compare for $a,b,c\le4$ (brute force) and $a,b,c\le8$ (determinant against product).

**P2.6 (★★★) The arctic circle.** Sample a uniformly random lozenge tiling of the regular hexagon of side $n=40$ by heat-bath Glauber dynamics on the heights of the plane partition (given its neighbours, a column height is uniform on the allowed interval; update the two colour classes of a checkerboard alternately). Draw it with three shades and overlay the inscribed circle. Then estimate, along a horizontal line through the centre, the fraction of each lozenge type, and compare with the frozen regions predicted by Cohn, Larsen and Propp.

**P2.7 (★★) Alternating sign matrices.** Enumerate the alternating sign matrices of size $n\le5$ (entries in $\{-1,0,1\}$, every row and column sums to $1$, and the non-zero entries alternate in sign along every row and column), and compare with $\prod_{j=0}^{n-1}\frac{(3j+1)!}{(n+j)!}$. How many of them are permutation matrices?

## Solutions

### P2.1
**How to find it.** Sum $\frac12(x_ky_{k+1}-x_{k+1}y_k)$ around the polygon. For Monte Carlo, a star-shaped polygon with vertices at angles $\theta_k$ and radii $r_k$ contains a point of angle $\theta$ if its radius is below the boundary at that angle, which is linear in between in the right parametrisation; simplest is a point-in-polygon test by ray casting.

In [ ]:
def shoelace(P):
    x, y = np.asarray(P, float).T
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)

print("Exercise 2.1:", shoelace([(0, 0), (4, 0), (5, 3), (2, 5), (-1, 2)]))

def inside(pt, P):
    x, y = pt; c = False
    for (x1, y1), (x2, y2) in zip(P, np.roll(P, -1, axis=0)):
        if (y1 > y) != (y2 > y) and x < x1 + (y - y1) * (x2 - x1) / (y2 - y1):
            c = not c
    return c

theta = np.sort(rng.uniform(0, 2*np.pi, 30)); r = rng.uniform(0.4, 1.0, 30)
P = np.c_[r * np.cos(theta), r * np.sin(theta)]
N = 20000; pts = rng.uniform(-1, 1, size=(N, 2))
mc = 4 * np.mean([inside(p, P) for p in pts])
print("shoelace %.4f   Monte Carlo %.4f (+- %.4f)" % (shoelace(P), mc, 4 * np.sqrt(mc/4 * (1 - mc/4) / N)))

### P2.2
**How to find it.** Index rows and columns of $C_k(A)$ by the $k$-subsets in lexicographic order (`itertools.combinations` produces exactly that order) and fill in the minors.

In [ ]:
def compound(A, k):
    m, n = A.shape
    R = list(itertools.combinations(range(m), k)); Cc = list(itertools.combinations(range(n), k))
    return np.array([[np.linalg.det(A[np.ix_(I, J)]) for J in Cc] for I in R])

A = rng.normal(size=(4, 6)); B = rng.normal(size=(6, 5))
for k in (1, 2, 3, 4):
    err = np.abs(compound(A @ B, k) - compound(A, k) @ compound(B, k)).max()
    print(f"k={k}: max |C_k(AB) - C_k(A)C_k(B)| = {err:.2e}")
M = rng.normal(size=(6, 3))
print("Pythagoras:", np.linalg.det(M.T @ M), sum(np.linalg.det(M[list(I)])**2 for I in itertools.combinations(range(6), 3)))
A = rng.normal(size=(5, 5)); lam = np.linalg.eigvals(A)
prods = np.sort_complex(np.array([np.prod(lam[list(I)]) for I in itertools.combinations(range(5), 2)]))
eig2 = np.sort_complex(np.linalg.eigvals(compound(A, 2)))
print("eigenvalues of C_2(A) vs products of pairs: max difference", np.abs(prods - eig2).max())

Sorting complex numbers can mismatch nearly equal real parts; if the last difference is not tiny, compare the two sets by a matching instead. With a random matrix it is fine.

### P2.3
**How to find it.** Keep three layers: $A^{(k-1)}$, $A^{(k)}$ and the new $A^{(k+1)}$; each new entry is a $2\times2$ determinant of neighbours in $A^{(k)}$ divided by the interior entry of $A^{(k-1)}$.

In [ ]:
def dodgson(A):
    A = [[Fraction(x) for x in row] for row in A]; n = len(A)
    prev = [[Fraction(1)] * (n + 1) for _ in range(n + 1)]; cur = A
    for size in range(n - 1, 0, -1):
        new = []
        for i in range(size):
            row = []
            for j in range(size):
                den = prev[i + 1][j + 1]
                if den == 0: raise ZeroDivisionError
                row.append((cur[i][j] * cur[i+1][j+1] - cur[i][j+1] * cur[i+1][j]) / den)
            new.append(row)
        prev, cur = cur, new
    return cur[0][0]

def exact_det(A):
    M = [[Fraction(x) for x in row] for row in A]; n = len(M); d = Fraction(1)
    for c in range(n):
        p = next((r for r in range(c, n) if M[r][c] != 0), None)
        if p is None: return Fraction(0)
        if p != c: M[c], M[p] = M[p], M[c]; d = -d
        d *= M[c][c]
        for r in range(c + 1, n):
            f = M[r][c] / M[c][c]
            M[r] = [a - f * b for a, b in zip(M[r], M[c])]
    return d

print("Exercise 2.7:", dodgson([[1,2,3,4],[2,1,2,3],[3,2,1,2],[4,3,2,1]]))
ok = fail = 0
for _ in range(200):
    n = int(rng.integers(2, 7)); A = rng.integers(-5, 6, size=(n, n)).tolist()
    try:
        assert dodgson(A) == exact_det(A); ok += 1
    except ZeroDivisionError:
        fail += 1
print("random integer matrices: agree", ok, " zero denominators", fail)

fails = 0; T = 2000
for _ in range(T):
    A = rng.integers(0, 2, size=(6, 6)).tolist()
    try: dodgson(A)
    except ZeroDivisionError: fails += 1
print("0/1 matrices of size 6: zero denominator in %.1f%% of cases" % (100 * fails / T))

def det_by_shifts(A):
    # det(A + tI) is a monic polynomial of degree n in t: interpolate n+1 values at large t.
    n = len(A); ts = [10**6 + 17 * k for k in range(n + 1)]
    vals = [dodgson([[A[i][j] + (t if i == j else 0) for j in range(n)] for i in range(n)]) for t in ts]
    # Lagrange interpolation at t = 0
    tot = Fraction(0)
    for k, tk in enumerate(ts):
        L = Fraction(1)
        for m, tm in enumerate(ts):
            if m != k: L *= Fraction(0 - tm, tk - tm)
        tot += vals[k] * L
    return tot

A = [[1, 1, 0], [1, 0, 1], [0, 1, 1]]
print("the failing example of Exercise 2.7:", det_by_shifts(A), "(exact:", exact_det(A), ")")

For $0/1$ matrices the algorithm almost always fails: a single zero among the $16$ interior entries of a $6\times6$ matrix already stops it at the third step, and the $2\times2$ minors of a $0/1$ matrix vanish often too. The shifted version never fails here, because the interior minors of $A+tI$ are polynomials in $t$ with leading coefficient $1$, non-zero for large $t$.

### P2.4
**How to find it.** Enumerate all north-east paths between two points as sequences of steps, then all choices of one path per source for each permutation; test vertex-disjointness with sets.

In [ ]:
def paths(a, b):
    (x0, y0), (x1, y1) = a, b
    if x1 < x0 or y1 < y0: return []
    E, Nn = x1 - x0, y1 - y0; out = []
    for east in itertools.combinations(range(E + Nn), E):
        x, y = x0, y0; pts = [(x, y)]; es = set(east)
        for s in range(E + Nn):
            if s in es: x += 1
            else: y += 1
            pts.append((x, y))
        out.append(pts)
    return out

def perm_sign(p):
    s = 1; p = list(p)
    for i in range(len(p)):
        for j in range(i + 1, len(p)):
            if p[i] > p[j]: s = -s
    return s

def lgv_check(A, B):
    k = len(A)
    M = np.array([[len(paths(a, b)) for b in B] for a in A])
    signed = 0
    for sigma in itertools.permutations(range(k)):
        for system in itertools.product(*[paths(A[i], B[sigma[i]]) for i in range(k)]):
            sets = [set(p) for p in system]
            if all(not (sets[i] & sets[j]) for i in range(k) for j in range(i + 1, k)):
                signed += perm_sign(sigma)
    return round(np.linalg.det(M)), signed

print(lgv_check([(0, 1), (1, 0)], [(3, 4), (4, 3)]))
print(lgv_check([(0, 2), (1, 0), (3, 0)], [(2, 5), (4, 4), (5, 2)]))
print(lgv_check([(0, 0), (2, 0)], [(3, 3), (1, 3)]))    # sinks in the 'wrong' order: negative count

In the last case the only non-intersecting systems use the transposition, and the determinant is negative: the sign in the lemma is real.

### P2.5
**How to find it.** (a) and (b) are formulas. For (c), build the plane partitions row by row: each row is a weakly decreasing sequence bounded entrywise by the row above.

In [ ]:
def T_det(a, b, c):
    M = [[comb(a + b, a + i - j) if 0 <= a + i - j <= a + b else 0 for j in range(c)] for i in range(c)]
    return exact_det(M)

def T_macmahon(a, b, c):
    r = Fraction(1)
    for i in range(1, a + 1):
        for j in range(1, b + 1):
            for k in range(1, c + 1):
                r *= Fraction(i + j + k - 1, i + j + k - 2)
    return r

def T_brute(a, b, c):
    def rows_below(top):
        def rec(j, prev):
            if j == b: yield (); return
            for v in range(min(prev, top[j]) + 1):
                for rest in rec(j + 1, v): yield (v,) + rest
        return list(rec(0, c))
    count = 0
    def go(i, top):
        nonlocal count
        if i == a: count += 1; return
        for r in rows_below(top): go(i + 1, r)
    go(0, (c,) * b)
    return count

assert all(T_det(a, b, c) == T_macmahon(a, b, c) == T_brute(a, b, c)
           for a in range(1, 5) for b in range(1, 5) for c in range(1, 5))
assert all(T_det(a, b, c) == T_macmahon(a, b, c) for a in range(1, 9) for b in range(1, 9) for c in range(1, 9))
print("all agree;  T(2,2,2) =", T_det(2, 2, 2), " T(3,3,3) =", T_det(3, 3, 3), " T(10,10,10) =", T_det(10, 10, 10))

### P2.6
**How to find it.** Heights $h_{ij}\in[0,n]$ of a plane partition in an $n\times n$ box, non-increasing in $i$ and $j$, with walls of height $n$ behind and floors of height $0$ in front. Cells of one checkerboard colour have neighbours only of the other colour, so they can all be resampled at once. Then project the visible faces of the cubes, as in the cover script.

In [ ]:
def sample_heights(n, sweeps, seed=0):
    g = np.random.default_rng(seed)
    h = np.full((n, n), n // 2)
    I, J = np.indices((n, n)); masks = [(I + J) % 2 == 0, (I + J) % 2 == 1]
    for _ in range(sweeps):
        for m in masks:
            p = np.pad(h, 1); p[0, :] = n; p[:, 0] = n
            up = np.minimum(p[:-2, 1:-1], p[1:-1, :-2]); lo = np.maximum(p[2:, 1:-1], p[1:-1, 2:])
            new = lo + (g.random((n, n)) * (up - lo + 1)).astype(int)
            h[m] = new[m]
    return h

def faces(h):
    n = h.shape[0]
    H = lambda i, j: n if i < 0 or j < 0 else (0 if i >= n or j >= n else h[i, j])
    cs = np.cos(np.pi / 6); P = lambda x, y, z: (cs * (x - y), z - 0.5 * (x + y))
    out = {0: [], 1: [], 2: []}
    for i in range(n):
        for j in range(n):
            z = h[i, j]; out[0].append([P(i, j, z), P(i+1, j, z), P(i+1, j+1, z), P(i, j+1, z)])
    for i in range(n + 1):
        for j in range(n):
            for z in range(H(i, j), H(i - 1, j)):
                out[1].append([P(i, j, z), P(i, j+1, z), P(i, j+1, z+1), P(i, j, z+1)])
    for j in range(n + 1):
        for i in range(n):
            for z in range(H(i, j), H(i, j - 1)):
                out[2].append([P(i, j, z), P(i+1, j, z), P(i+1, j, z+1), P(i, j, z+1)])
    return out

n = 40
h = sample_heights(n, 4000)
F = faces(h)
fig, ax = plt.subplots(figsize=(7, 7))
for t, col in zip(range(3), ['#f4d58d', '#bf8b2e', '#5b3a12']):
    ax.add_collection(PolyCollection(F[t], facecolors=col, edgecolors='none'))
cy = 0.5 * n * 0  # centre of the hexagon in these coordinates
ax.add_patch(plt.Circle((0, 0), n * np.cos(np.pi / 6), fill=False, color='crimson', lw=1.5))
ax.set_xlim(-n, n); ax.set_ylim(-n, n); ax.set_aspect('equal'); ax.axis('off')
plt.show()
print("lozenges of each kind:", [len(F[t]) for t in range(3)], " (n^2 =", n * n, ")")

The three counts are all $n^2=1600$, as the problem of the calissons (Exercise 2.12) predicts for every tiling. The circle inscribed in the hexagon separates the frozen corners (a single shade) from the disordered middle. The hexagon's centre is at the origin of the projection because the box is a cube.

In [ ]:
# Fraction of each type along the horizontal line y = 0, in bins of x.
centres = {t: np.array([np.mean(np.array(f), axis=0) for f in F[t]]) for t in range(3)}
bins = np.linspace(-n * np.cos(np.pi/6), n * np.cos(np.pi/6), 25)
mid = 0.5 * (bins[1:] + bins[:-1])
counts = np.array([np.histogram(c[np.abs(c[:, 1]) < 2, 0], bins)[0] for c in centres.values()], float)
frac = counts / np.maximum(counts.sum(axis=0), 1)
for t, lab in zip(range(3), ['top faces', 'left faces', 'right faces']):
    plt.plot(mid / (n * np.cos(np.pi/6)), frac[t], label=lab)
plt.axvline(-1, color='gray', ls=':'); plt.axvline(1, color='gray', ls=':')
plt.xlabel('x / (distance to the vertical sides)'); plt.ylabel('fraction'); plt.legend(); plt.show()

On the horizontal line through the centre the inscribed circle touches the two vertical sides, so the whole segment is in the disordered region; the fractions of the three kinds vary smoothly across it. Lines higher or lower cross the frozen corners (try $|y|$ near $0.8n$).

### P2.7
**How to find it.** Build the matrix row by row. A row of an ASM is determined by the positions of its $+1$s and $-1$s, alternating and starting and ending with $+1$; the column condition says that the partial column sums stay in $\{0,1\}$ and end at $1$. So keep the vector of partial column sums, a $0/1$ vector, and choose each new row compatibly: where the partial sum is $1$ the entry can be $0$ or $-1$, where it is $0$ the entry can be $0$ or $+1$, with alternation along the row.

In [ ]:
def count_asm(n):
    from functools import lru_cache
    @lru_cache(None)
    def rows(state):
        # all rows compatible with the partial column sums 'state' (a tuple of 0/1)
        out = []
        def rec(j, last, row):
            if j == n:
                if last == 1: out.append(tuple(row))
                return
            rec(j + 1, last, row + [0])
            if state[j] == 0 and last in (0, -1): rec(j + 1, 1, row + [1])
            if state[j] == 1 and last == 1: rec(j + 1, -1, row + [-1])
        rec(0, 0, [])
        return out
    @lru_cache(None)
    def count(i, state):
        if i == n: return 1 if all(s == 1 for s in state) else 0
        return sum(count(i + 1, tuple(s + r for s, r in zip(state, row))) for row in rows(state))
    return count(0, (0,) * n)

formula = lambda n: math.prod(math.factorial(3*j + 1) for j in range(n)) // math.prod(math.factorial(n + j) for j in range(n))
for n in range(1, 8):
    print(n, count_asm(n), formula(n), " permutation matrices:", math.factorial(n))

The counts $1,2,7,42,429,7436,218348$ match the Mills–Robbins–Rumsey formula proved by Zeilberger. Permutation matrices are the ASMs without $-1$; they are a vanishing fraction.